# FleetLogix | Avance #1 — Generación y calidad de datos

Notebook de presentación y auditoría **no destructiva**. El esquema y los scripts originales se conservan. No ejecutar la carga nuevamente si la base ya está poblada.

**Objetivo:** 505.650 registros entre seis tablas, con integridad y consistencia temporal.

## 1. Configuración y trazabilidad
La generación original se realiza con `Scripts/01_data_generation.py` y `Scripts/02_generate_transactions.py`. La carga original se realiza con `Scripts/03_load_all.py --load` solo en una base vacía. Este notebook conecta en modo de lectura y prepara evidencia visual.

In [ ]:
from pathlib import Path
import os
import pandas as pd
import matplotlib.pyplot as plt
from dotenv import load_dotenv
import psycopg2

ROOT = Path.cwd()
if not (ROOT / "Scripts").is_dir() and (ROOT.parent / "Scripts").is_dir():
    ROOT = ROOT.parent
load_dotenv(ROOT / ".env")
con = psycopg2.connect(host=os.getenv("PGHOST", "localhost"), port=os.getenv("PGPORT", "5432"),
    dbname=os.getenv("PGDATABASE", "fleetlogix_db"), user=os.getenv("PGUSER", "postgres"),
    password=os.getenv("PGPASSWORD"))
con.set_session(readonly=True, autocommit=True)
def consultar(sql):
    with con.cursor() as cur:
        cur.execute(sql)
        columns = [d[0] for d in cur.description]
        return pd.DataFrame(cur.fetchall(), columns=columns)
consultar("SELECT current_database() AS base, current_user AS usuario")

## 2. Conteos por tabla

In [ ]:
conteos = consultar("""SELECT 'vehicles' tabla, count(*) registros FROM vehicles UNION ALL
SELECT 'drivers', count(*) FROM drivers UNION ALL SELECT 'routes', count(*) FROM routes UNION ALL
SELECT 'trips', count(*) FROM trips UNION ALL SELECT 'deliveries', count(*) FROM deliveries UNION ALL
SELECT 'maintenance', count(*) FROM maintenance""")
objetivo = {'vehicles':200,'drivers':400,'routes':50,'trips':100000,'deliveries':400000,'maintenance':5000}
conteos['esperado'] = conteos['tabla'].map(objetivo)
conteos['correcto'] = conteos['registros'] == conteos['esperado']
assert conteos['correcto'].all(), 'Al menos una tabla no tiene el número esperado'
display(conteos)
print('Total:', conteos['registros'].sum())

## 3. Relaciones, rangos y capacidad

In [ ]:
fk = consultar("""SELECT count(*) FILTER(WHERE v.vehicle_id IS NULL) vehiculos_invalidos,
 count(*) FILTER(WHERE d.driver_id IS NULL) conductores_invalidos,
 count(*) FILTER(WHERE r.route_id IS NULL) rutas_invalidas
 FROM trips t LEFT JOIN vehicles v USING(vehicle_id)
 LEFT JOIN drivers d USING(driver_id) LEFT JOIN routes r USING(route_id)""")
calidad = consultar("""SELECT count(*) FILTER(WHERE arrival_datetime IS NULL OR arrival_datetime<=departure_datetime) AS tiempos_invalidos,
 min(departure_datetime) AS primer_viaje, max(departure_datetime) AS ultima_salida,
 max(arrival_datetime) AS ultima_llegada FROM trips""")
display(fk, calidad)
assert (fk.iloc[0, :3] == 0).all()
assert calidad.iloc[0]['tiempos_invalidos'] == 0

## 4. Distribución de entregas por viaje
La distribución es deliberadamente exacta: 5 %, 20 %, 50 %, 20 % y 5 % para 2, 3, 4, 5 y 6 entregas.

In [ ]:
entregas = consultar("""WITH n AS (SELECT t.trip_id,count(d.delivery_id) AS entregas FROM trips t
LEFT JOIN deliveries d USING(trip_id) GROUP BY t.trip_id)
SELECT entregas, count(*) AS viajes FROM n GROUP BY entregas ORDER BY entregas""")
display(entregas)
assert entregas['entregas'].between(2,6).all()
assert int((entregas['entregas']*entregas['viajes']).sum()) == 400000
ax=entregas.plot.bar(x='entregas',y='viajes',legend=False,figsize=(8,3.5))
ax.set(title='Distribución de entregas por viaje',xlabel='Entregas por viaje',ylabel='Número de viajes')
plt.tight_layout(); plt.show()

## 5. Frecuencia de mantenimiento
Se generan 500 viajes y 25 mantenimientos por vehículo, por diseño. Esto prueba una razón **agregada** 20:1; no demuestra que todos los intervalos reales entre eventos de mantenimiento sean exactamente 20 viajes.

In [ ]:
mant = consultar("""WITH t AS (SELECT vehicle_id,count(*) viajes FROM trips GROUP BY vehicle_id),
m AS (SELECT vehicle_id,count(*) mantenimientos FROM maintenance GROUP BY vehicle_id)
SELECT v.vehicle_id,t.viajes,m.mantenimientos, round(t.viajes::numeric/nullif(m.mantenimientos,0),2) relacion
FROM vehicles v JOIN t USING(vehicle_id) JOIN m USING(vehicle_id) ORDER BY v.vehicle_id""")
display(mant.describe())
assert (mant['viajes']==500).all() and (mant['mantenimientos']==25).all()

## 6. Cobertura mensual y estacionalidad
Distinguir los meses extremos parciales y la variación horaria de la demanda. No interpretar el ajuste de pesos de fines de semana como una reducción automática de viajes diarios.

In [ ]:
meses = consultar("""SELECT date_trunc('month',departure_datetime)::date mes,
count(*) AS viajes, round(avg(extract(epoch FROM (arrival_datetime-departure_datetime))/3600)::numeric,2) AS horas_promedio
FROM trips GROUP BY 1 ORDER BY 1""")
display(meses)
plt.figure(figsize=(11,4)); plt.plot(meses['mes'],meses['viajes'],marker='o');
plt.title('Viajes registrados por mes');plt.ylabel('Viajes');plt.xticks(rotation=45);plt.tight_layout();plt.show()

## 7. Registros de ejecución y evidencias
Revisar `logs/generacion_transacciones.log` y conservar las capturas / salidas de `03_load_all.py`. El log de generación confirma la preparación de archivos, **no** por sí solo la carga exitosa a PostgreSQL.

In [ ]:
log = ROOT/'logs'/'generacion_transacciones.log'
if log.exists():
    print('\n'.join(log.read_text(encoding='utf-8').splitlines()[-12:]))
else:
    print('El log no está en esta carpeta; copiar la evidencia original al proyecto.')

## 8. Metodología de generación (resumen)

`gethourly_distribution(weekday)` construye 24 pesos, incrementa ventanas operativas, ajusta ciertas franjas en fines de semana y los normaliza para que sumen 1.

`generate_trips(vehicles, routes, rng)` recorre 200 vehículos y 500 viajes por vehículo. Asigna dos conductores exclusivos por vehículo; alterna sus identificadores, selecciona rutas con origen en la ciudad actual, controla una separación mínima de 2 h después de la llegada anterior, calcula duración y combustible, y crea 25 mantenimientos por vehículo, uno tras cada 20 viajes. Los pesos se incorporan después, en `generate_deliveries`.

**Límites:** una pareja fija de conductores por vehículo, 500 viajes exactos por vehículo, 25 mantenimientos exactos por vehículo, tasa de rutas interurbanas de 3,5 % al seleccionar, y todas las entregas con estado `delivered`. Son simplificaciones sintéticas, no evidencia de comportamientos medidos.

## 9. Cierre
El Avance #1 cubre carga, relaciones y controles principales. Antes de la entrega, integrar el diagrama ER exportado por DBeaver, el PDF de análisis, el diccionario y logs auténticos. **No subir `.env` ni archivos con credenciales**.

In [ ]:
con.close()